Automatic Differentiation (autoGrad)
PyTorch automatically computes every derivative.
You only define
    the network,
    the loss.

PyTorch computes all gradients.
This is called Automatic Differentiation (Autograd).
let's take a example 
f(x)=x**2 
so df/dx=2*x

if x = 3 ,f(2)=3 and df(2)= 6 so without autograd you will do all this manually 
the next code ulstrate the example 

In [1]:
import torch
x = torch.tensor(3.0,requires_grad=True)
y=x**2
y.backward()
print(x.grad)

tensor(6.)


what does requires_grad = true mean ? 
# x=torch.tensor(3.0, requires_grad=True)
this tells pytorch : 
    trach every operation involving this tensor because i will later ask for gradients ,without it it will ignore gradient computation 


the computational Graph

In [ ]:
#example 
x = torch.tensor(2.0, requires_grad=True)
y = x+3
z =y*y

for this pytorch will buils a graph 

    x
    ↓
    +3
    ↓
    y
    ↓
    Square
    ↓
    z
==>every operation becomes a node 
internallly 
    Tensor
    ↓
    Operation
    ↓
    Tensor
    ↓
    Operation
    ↓
    Tensor

so the graph stores 
    inputs,
    outputs,
    operations,
    how to compute derivatives

In [7]:
import torch
x = torch.tensor(2., requires_grad=True)

y = x + 3
z = y**2
print(z)

tensor(25., grad_fn=<PowBackward0>)


# call backward() z.backward
pytorch now walks backwaard throught the graph
    x
    ↓
    +3
    ↓
    y
    ↓
    Square
    ↓
    z

backward 
    z
    ↑
    Square
    ↑
    y
    ↑
    +3
    ↑
    x

using the chai rule, it computes dz/dx                              

In [6]:
z.backward()
print(x.grad)

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

This means that PyTorch thinks you have already called backward() once on this computation graph, and now you're trying to call it again after the graph has been freed.
beacuase we use jupter the graphe i already exected from the previous cells 
suppose earlier you alerady exected 
    z.backward()
then later you execute again
    z.backward()

the computation graph has alerady been destroyed 
why does pytroch destory the graph 
    autograd stores intermediate values  for the graphe x--- + ---y --square -----z 
    during the forward pass it saves x =2,y=5 ,z= 25 (requires_grad = true )
    when you call z.backward it uses those saved value , after finishing deletes them to save memory


In [8]:
import torch

x = torch.tensor(2., requires_grad=True)

y = x + 3
z = y**2

z.backward()

print(x.grad)

tensor(10.)


In [9]:
z.backward()

print(x.grad)

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
#how tell pytorch to tell the graph 
z.backward(retain_graph=True) 
# which const you a lot of memory be careful bro 

another exemple for the autograd 

In [ ]:
x = torch.tensor(4.,requires_grad=True)
y = 3*x
z = y+5
loss = z**2

In [14]:
import torch 
import torch.nn as nn 
x = torch.tensor(5.)
w= torch.tensor(2.,requires_grad=True)
b= torch.tensor(1.,requires_grad=True)

target = torch.tensor(15.)


z = w*x + b
#z = y.Tanh() this will give you error because tanh is not a function it a class ,pyton he will
#that you create tanh object and you pass y to its constractor (tanh.__init__(self,y))
#but the constructor og tanh dosen't expect any argements besides self which make error 
#the correct way using nn.module first create the activation layer tanh= nn.tanh()
#then apply it oro your tensor 

tanh = nn.Tanh()
#z = tanh(y)

loss = 0.5*(z - target)**2
loss.backward()
print(w.grad)
print(b.grad)

tensor(-20.)
tensor(-4.)


loss.backward() actually does not update the weight it only computes the gradient 
    Forward
    ↓
    Loss
    ↓
    backward()
    ↓
    Compute gradients
    ↓
    Store them in

    parameter.grad

the optimizer performs the update 


In [ ]:
import torch
import torch.nn as nn 

#input and target 
x = torch.tensor([[2.0]])
target = torch.tensor([[8.0]])


#linear layer 
model = nn.Linear(1,1)
#loss function
criterion = nn.MSELoss()
#optimizer 
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)
#forward pass 
prediction = model(x)

loss = criterion(prediction,target)

#compute gradients
loss.backward()

#update parameters 
optimizer.step()

#clear gradients for the next iteration 
optimizer.zero_grad()



tensor(-20.)
tensor(-4.)


why optimizer.zero_grad()
gradients accmulate by default in pytorch
iteration 1 gradient is 2
iteration 2 gradient is 3 
so without clearning it will stored 5 

Input
   │
   ▼
Neural Network
   │
   ▼
Prediction
   │
   ▼
Loss Function
   │
   ▼
loss.backward()
   │
   ▼
torch.autograd computes all gradients
   │
   ▼
Optimizer updates weights
   │
   ▼
Better neural network

torch.autograd is PyTorch's automatic differentiation engine.
Setting requires_grad=True tells PyTorch to track operations on a tensor.
Every operation creates a computational graph.
loss.backward() traverses this graph backward and computes gradients using the chain rule.
The gradients are stored in each parameter's .grad attribute.
optimizer.step() uses those gradients to update the model parameters.
optimizer.zero_grad() clears old gradients before the next training iteration.

In short:

Forward pass computes the predictions and the loss.
Autograd computes how each parameter influenced that loss.
The optimizer uses those gradients to improve the model.

Disabling gradient tracking 
By default, all tensors with requires_grad=False are tracking their computational history and support gradient computation(Since none of the inputs require gradients, PyTorch doesn't build a computation graph.). However, there are some cases when we do not need to do that, for example, when we have trained the model and just want to apply it to some input data, i.e. we only want to do forward computations through the network. We can stop tracking computations by surrounding our computation code with torch.no_grad() block:

In [22]:
x= torch.tensor(1.)
w= torch.tensor(2.,requires_grad=True)
b= torch.tensor(5., requires_grad=True)
z= x*w+b
print(z.requires_grad)

with torch.no_grad():
    z= w*x+b
print(z.requires_grad)

True
False


PyTorch determines requires_grad as follows:

If at least one input has requires_grad=True and you're not inside torch.no_grad(), then the output will have requires_grad=True.
If all inputs have requires_grad=False, the output is False.
Inside torch.no_grad(), the output is always False, regardless of the inputs.
another way to achieve the same result is to use the detach() method on the tensor 
    z = torch.matmul(x, w)+b
    z_det = z.detach()
    print(z_det.requires_grad)
🎼🎵🎶🎵🎶🎵
H.B.D.M

why is better sometimes to disable the gradient traking 
    To mark some parameters in your neural network as frozen parameters.
    To speed up computations when you are only doing forward pass, because computations on tensors that do not track gradients would be more efficient.

More on computational graphe 
Conceptually, autograd keeps a record of data (tensors) and all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

run the requested operation to compute a resulting tensor

maintain the operation’s gradient function in the DAG.

The backward pass kicks off when .backward() is called on the DAG root. autograd then:

computes the gradients from each .grad_fn,

accumulates them in the respective tensor’s .grad attribute

using the chain rule, propagates all the way to the leaf tensors.

This is one of the most advanced concepts in torch.autograd, but it's also one of the most important for understanding how PyTorch computes gradients efficiently.

The confusing part is that there are two different situations:

The output is a scalar (one number) → ordinary gradient.
The output is a vector or tensor → Jacobian and Jacobian-vector products.

Let's build the intuition step by step.
Why Doesn't PyTorch Compute the Whole Jacobian?
imagine input 10 M 
        output 10 M
    the jacobian size becomes 100 trillion 

nstead PyTorch computes

Jacobian-Vector Product
Instead of computing J
pytorch computes transpose(v)* J
    using the chain rule, without ever constructing the full Jacobian.
    This is much cheaper in both memory and computation.




In [31]:
import torch
from torch.autograd import backward

x = torch.tensor(2., requires_grad=True)

y = torch.tensor([
    x**2,
    3*x
])

print(y)
y.backward() # this will give you error because y is not a scalar value

tensor([4., 6.])


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

this happen because 
Which gradient do you want?

y₁ ?

y₂ ?

Both?

solution provide the vector v 

In [32]:
import torch

x = torch.tensor(2., requires_grad=True)

y = torch.stack([
    x**2,
    3*x
])

v = torch.tensor([2.,5.])

y.backward(v)

print(x.grad)

tensor(23.)


Why Does backward(v) Work? 
pytorch interprets it as :
"compute the weight sum of the output using v ,tne diferntiate that sum "

Relation to Neural Networks
suppose a network outputs [cat, dog, bird] there are logits .
we don not call `output.backward()`
instead we compute a scalar loss:`loss = CrossEntropy(output, target)`
now output --> loss --> one sclar 
so calling loss.backward() automatically computes dL/dw fo every parater 
    That's why, during standard deep learning training, you almost never need to manually supply the vector v

When Is Jacobian-Vector Product Used?
Although uncommon in basic training loops, it is very important in advanced techniques:

    Meta-learning (e.g., MAML)
    Second-order optimization (Newton, Hessian-free methods)
    Physics-informed neural networks (PINNs)
    Implicit differentiation
    Sensitivity analysis
    Scientific computing
    Custom autograd functions

| Output Type | Example                 | What `backward()` Computes                                                              |
| ----------- | ----------------------- | --------------------------------------------------------------------------------------- |
| Scalar      | `loss = 3.5`            | Ordinary gradient (\nabla_x,\text{loss})                                                |
| Vector      | `y = [y₁, y₂, ..., yₘ]` | Requires a vector `v`; computes a Jacobian-vector product rather than the full Jacobian |


for more check :https://docs.pytorch.org/docs/2.12/notes/autograd.html